# Kaggle: Jailbreak — тяжёлые пресеты AutoIntent (classic-medium, nn, zero-shot, BERT FT)

**Задание (формулировка):** AutoIntent с более тяжёлыми пресетами — **classic-medium**, **nn**, **zero-shot**, **BERT fine-tuning**. В этом ноутбуке это реализовано как пресеты AutoIntent 0.2.0: `classic-medium`, `nn-medium` (вариант «nn»), `zero-shot-encoders` (вариант «zero-shot» без LLM API), `transformers-light` (BERT / transformer fine-tuning search space). Дальше по метрикам и логам решаешь, нужно ли углубляться в дообучение эмбеддера.

**Цель:** сравнить search-space пресеты **autointent==0.2.0** на WildJailbreak и по логам решить, есть ли смысл углубляться в дообучение эмбеддера.

**Пресеты в цикле (по умолчанию):**
| Запрос | Имя в AutoIntent | Комментарий |
|--------|------------------|-------------|
| classic-medium | `classic-medium` | тяжелее `classic-light` |
| nn | `nn-medium` | нейросетевой search space (есть ещё `nn-heavy`) |
| zero-shot | `zero-shot-encoders` | энкодерный zero-shot; **`zero-shot-llm`** требует LLM API — не включён по умолчанию |
| BERT fine-tuning | `bert-finetune` (→ `transformers-light`) | transformer/BERT-style fine-tuning search space |

**Данные:** сначала `prepare_data.py` без флагов создаёт raw, few-shot `10/20/50 × 42/123/456`, `test.json`, `wildjailbreak_eval_binary.jsonl`; затем `prepare_data.py --full_subset` добавляет `wildjailbreak_full100k_seed{42,123,456}.json`. Ячейка `5b` проверяет наличие и согласованность файлов до обучения.

**Runner:** прогоны идут через `run_autointent_logged.py` — thin wrapper вокруг `run_autointent.py` с максимальным логированием для Kaggle output. В **§6a** и **§6c** — отдельные ячейки few-shot и full; **§6b** / **§6d** печатают срезы `metrics.json`; дочерний Python с `-u`, чтобы лог не залипал в буфере.

**Логи в output / `metrics.json`:** полный `METRICS_JSON` на каждый прогон: core metrics, subset recalls, TP/FP/FN/TN, FNR/FPR, data summaries, prediction breakdown by `data_type`, raw `None` predictions, score summary, decision attrs, HF embedder id, runtime/package versions, train/eval timings, run config, artifact manifest. `HYPOTHESIS_LOG` дублирует ключевые поля для быстрого разбора гипотезы.

**Fine-tuning / embedder:** `transformers-light` включает transformer/BERT-style fine-tuning search space. `NO_FIX_EMBEDDER=True` даёт AutoIntent выбирать embedder и логирует выбранный `embedder_hf_model`; это основной режим для ответа «надо ли лезть глубже в эмбеддер». Для threshold/asymmetric cost всё равно нужен отдельный val threshold sweep.

**Время:** полная сетка = **48** прогонов всего: 4 пресета × (3 shots × 3 сида few-shot + 3 full на сиды). Очень долго. Для smoke уменьшите `PRESETS` или пропустите ячейку **§6c** (full).

**Артефакты на Kaggle:** `/kaggle/working` живёт только в рамках сессии ядра; после остановки или таймаута файлы могут пропасть. Включи в настройках ноутбука **Persistence** (если доступно), при **Save Version** отметь сохранение **output**, и обязательно выполни **§8** — там архив всего `PROJECT_ROOT` и ссылка на скачивание zip. Во время **§6a** и **§6c** актуальный **`metrics.json`** копируется в **`/kaggle/working/metrics_jailbreak_latest.json`** после каждого прогона.


## 1. Пути


In [ ]:
import os, subprocess, sys
from pathlib import Path
IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
PROJECT_ROOT = WORK / "kaggle_jailbreak_heavy_presets"
TASK_ROOT = PROJECT_ROOT / "tasks" / "jailbreak_detection"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


## 2. pip


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "transformers==4.49.0", "datasets==3.6.0", "sentence-transformers==3.4.1",
    "autointent==0.2.0", "scikit-learn>=1.5,<2", "pandas>=2.0", "numpy>=1.24,<3",
    "matplotlib>=3.7", "seaborn>=0.12", "optuna>=4,<5", "wandb>=0.19,<1",
    "python-dotenv>=1,<2"], check=True)


## 3. Записать код

Ячейка ниже раскладывает файлы под `PROJECT_ROOT` из **base64** (иначе при разбиении длинной строки `json.loads('…')` на фрагменты Kaggle/Jupyter даёт `SyntaxError: unterminated string literal`). После правок в репозитории пересобери ячейку:  
`python3 tasks/jailbreak_detection/notebooks/rebuild_kaggle_cell6_payload.py` из корня репозитория.


In [ ]:
import base64
import json
from pathlib import Path

# Файлы проекта в base64: иначе при разбиении ячейки на куски ломается литерал json.loads('…').
_BLOB = """
...
"""

_P = json.loads(base64.b64decode(_BLOB.encode("ascii")).decode("utf-8"))

for rel, txt in _P.items():
    p = PROJECT_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt, encoding="utf-8")
print("Written", len(_P), "files")


In [ ]:
from textwrap import dedent

logged_runner = r'''
from __future__ import annotations

import argparse
import importlib.metadata as importlib_metadata
import json
import os
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# При запуске как `python /abs/path/.../run_autointent_logged.py` первый элемент sys.path —
# каталог скрипта, а не корень репозитория; без этого пакет `tasks` не находится.
_PROJECT_ROOT = Path(__file__).resolve().parents[3]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np

from tasks.jailbreak_detection.scripts import run_autointent as base


def _file_info(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"path": str(path), "exists": False}
    st = path.stat()
    return {
        "path": str(path),
        "exists": True,
        "bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat(),
    }


def _text_stats(texts: list[str]) -> dict[str, Any]:
    lens = np.asarray([len(x) for x in texts], dtype=np.float64)
    if len(lens) == 0:
        return {"n": 0}
    return {
        "n": int(len(texts)),
        "chars_mean": float(np.mean(lens)),
        "chars_std": float(np.std(lens)),
        "chars_min": int(np.min(lens)),
        "chars_p50": float(np.percentile(lens, 50)),
        "chars_p95": float(np.percentile(lens, 95)),
        "chars_max": int(np.max(lens)),
    }


def _train_summary(train_raw: dict, source_file: Path) -> dict[str, Any]:
    safe = []
    for intent in train_raw.get("intents", []):
        safe.extend(intent.get("utterances", []))
    jailbreak = list(train_raw.get("oos_utterances", []))
    total = len(safe) + len(jailbreak)
    return {
        "source_file": _file_info(source_file),
        "n_total": total,
        "n_safe": len(safe),
        "n_jailbreak": len(jailbreak),
        "class_balance_safe": float(len(safe) / total) if total else None,
        "class_balance_jailbreak": float(len(jailbreak) / total) if total else None,
        "safe_text_stats": _text_stats(safe),
        "jailbreak_text_stats": _text_stats(jailbreak),
    }


def _test_summary(test_raw: dict, source_file: Path) -> dict[str, Any]:
    texts = list(test_raw.get("utterances", []))
    labels = np.asarray(test_raw.get("labels", []))
    return {
        "source_file": _file_info(source_file),
        "n_total": len(texts),
        "n_safe": int(np.sum(labels == 0)),
        "n_jailbreak": int(np.sum(labels == 1)),
        "text_stats": _text_stats(texts),
    }


def _eval_summary(rows: list[dict], source_file: Path) -> dict[str, Any]:
    by_label, by_type = {}, {}
    for row in rows:
        by_label[row.get("binary_label")] = by_label.get(row.get("binary_label"), 0) + 1
        by_type[row.get("data_type")] = by_type.get(row.get("data_type"), 0) + 1
    return {
        "source_file": _file_info(source_file),
        "n_total": len(rows),
        "by_binary_label": dict(sorted(by_label.items())),
        "by_data_type": dict(sorted(by_type.items())),
    }


def _runtime() -> dict[str, Any]:
    packages = {}
    for pkg in ["autointent", "torch", "torchvision", "torchaudio", "transformers", "sentence-transformers", "datasets", "numpy", "scikit-learn", "optuna"]:
        try:
            packages[pkg] = importlib_metadata.version(pkg)
        except importlib_metadata.PackageNotFoundError:
            packages[pkg] = None
    return {
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cwd": str(Path.cwd()),
        "env": {k: os.environ.get(k) for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "LOKY_MAX_CPU_COUNT", "TOKENIZERS_PARALLELISM", "CUDA_VISIBLE_DEVICES"]},
        "kaggle": {"is_kaggle": Path("/kaggle/working").exists(), "working_exists": Path("/kaggle/working").exists(), "input_exists": Path("/kaggle/input").exists()},
        "packages": packages,
    }


def _limited(value: Any, max_chars: int = 12000) -> Any:
    try:
        dumped = json.dumps(value, ensure_ascii=False)
    except (TypeError, ValueError):
        return {"_error": "non_serializable"}
    if len(dumped) <= max_chars:
        return value
    if isinstance(value, dict):
        return {"_truncated": True, "json_chars": len(dumped), "top_level_keys": list(value.keys())}
    return {"_truncated": True, "json_chars": len(dumped), "type": type(value).__name__}


def _prediction_by_type(y_true: np.ndarray, y_pred: np.ndarray, data_types: np.ndarray) -> dict[str, Any]:
    out = {}
    for dtype in sorted({str(x) for x in data_types.tolist()}):
        mask = data_types == dtype
        yt, yp = y_true[mask], y_pred[mask]
        n = int(np.sum(mask))
        pred_jb = int(np.sum(yp == 1))
        out[dtype] = {
            "n": n,
            "true_safe": int(np.sum(yt == 0)),
            "true_jailbreak": int(np.sum(yt == 1)),
            "pred_safe": int(np.sum(yp == 0)),
            "pred_jailbreak": pred_jb,
            "pred_jailbreak_rate": float(pred_jb / n) if n else None,
            "correct": int(np.sum(yt == yp)),
            "accuracy": float(np.mean(yt == yp)) if n else None,
        }
    return out


def _artifact_manifest(model_dir: Path) -> dict[str, Any]:
    if not model_dir.exists():
        return {"model_dir": str(model_dir), "exists": False}
    top = []
    for p in sorted(model_dir.iterdir(), key=lambda x: x.name):
        row = {"name": p.name, "type": "dir" if p.is_dir() else "file"}
        if p.is_file():
            row["bytes"] = p.stat().st_size
        top.append(row)
    json_files = []
    for p in sorted(model_dir.rglob("*.json"))[:80]:
        info = _file_info(p)
        info["relative_path"] = str(p.relative_to(model_dir))
        json_files.append(info)
    return {"model_dir": str(model_dir), "exists": True, "top_level_entries": top, "json_files": json_files}


_ORIG_TRAIN = base.train


def train(args, data_dir: Path, model_dir: Path) -> None:
    started = time.perf_counter()
    _ORIG_TRAIN(args, data_dir, model_dir)
    meta_path = model_dir / "train_metadata.json"
    metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    if args.mode == "full":
        train_file = data_dir / base.full_train_filename(args.seed)
        train_raw = base.load_full_train(args.seed, data_dir)
    else:
        train_file = data_dir / f"train_shot{args.n_shots}_seed{args.seed}.json"
        train_raw = base.load_fewshot_train(args.n_shots, args.seed, data_dir)
    test_file = data_dir / "test.json"
    test_raw = base.load_test(data_dir)
    metadata["data_summary"] = {"train": _train_summary(train_raw, train_file), "test": _test_summary(test_raw, test_file)}
    metadata["run_config"] = {
        "mode": args.mode,
        "n_shots": args.n_shots if args.mode == "fewshot" else None,
        "seed": args.seed,
        "preset": args.preset,
        "pilot": args.pilot,
        "no_fix_embedder": args.no_fix_embedder,
        "data_config": {"scheme": "cv", "n_folds": 3} if args.mode == "fewshot" else {"scheme": "default_autointent"},
        "logging_config": {"project_dir": str(model_dir), "dump_modules": True, "clear_ram": False},
    }
    metadata["runtime_environment"] = _runtime()
    metadata["timings"] = {"train_total_sec": float(time.perf_counter() - started)}
    meta_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Enriched train_metadata.json with data_summary/run_config/runtime/timings")


def evaluate(args, data_dir: Path, model_dir: Path, results_dir: Path, runs_dir: Path | None = None) -> None:
    started = time.perf_counter()
    print("\n" + "=" * 60)
    print("AutoIntent Evaluation (Jailbreak Detection, MAX LOGGING)")
    print("=" * 60)
    metadata_path = model_dir / "train_metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {
        "model_name": base.get_model_name(args.pilot, args.no_fix_embedder, preset=args.preset),
        "mode": "full" if args.mode == "full" else f"{args.n_shots}shot",
        "n_shots": None if args.mode == "full" else args.n_shots,
        "seed": args.seed,
        "embedder": "auto (optimized by AutoML)" if args.no_fix_embedder else base.get_embedder_name(args.pilot),
        "embedder_fixed": not args.no_fix_embedder,
        "pilot": args.pilot,
        "preset": args.preset,
    }
    hf_emb = metadata.get("embedder_hf_model") or base.embedder_hf_model_from_dump(model_dir)
    print("Model:", metadata["model_name"])
    print("Mode:", metadata["mode"])
    print("Preset:", metadata.get("preset"))
    print("Embedder:", metadata.get("embedder"))
    print("HF embedder:", hf_emb)

    pipeline = base.Pipeline.load(model_dir)
    test_file = data_dir / "test.json"
    eval_file = data_dir / "wildjailbreak_eval_binary.jsonl"
    test_raw = base.load_test(data_dir)
    test_texts = test_raw["utterances"]
    eval_binary = base.load_eval_binary(data_dir)
    y_true = np.array([1 if r["binary_label"] == "jailbreak" else 0 for r in eval_binary])
    data_types = np.array([r["data_type"] for r in eval_binary])
    alignment = {"test_utterances": len(test_texts), "test_labels": len(test_raw.get("labels", [])), "eval_binary_rows": len(eval_binary)}
    alignment["aligned_lengths"] = alignment["test_utterances"] == alignment["test_labels"] == alignment["eval_binary_rows"]
    print("Data alignment:", alignment)
    if not alignment["aligned_lengths"]:
        raise ValueError(f"Test/eval data length mismatch: {alignment}")

    raw_preds = pipeline.predict(test_texts)
    raw_pred_values, raw_none = {}, 0
    for pred in raw_preds:
        if pred is None:
            raw_none += 1
        key = "None" if pred is None else str(pred)
        raw_pred_values[key] = raw_pred_values.get(key, 0) + 1
    y_pred = np.array([1 if p is None else p for p in raw_preds])
    metrics = base.evaluate_jailbreak(y_true, y_pred, data_types, oos_label=1)
    eval_counts = base.confusion_and_rates_jailbreak_positive(y_true, y_pred, positive_label=1)
    scores_eval_summary = base.scoring_eval_summary_from_pipeline(pipeline, test_texts)
    decision_attrs = _limited(base.decision_module_attrs_from_dump(model_dir))

    print("\n" + "=" * 60)
    print("Results")
    print("=" * 60)
    for k, v in metrics.items():
        if v is not None:
            print(f"  {k}: {v:.4f}")

    result = {
        "model_name": metadata["model_name"],
        "mode": metadata["mode"],
        "n_shots": metadata["n_shots"],
        "seed": metadata["seed"],
        "f1": metrics["f1"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "over_refusal_rate": metrics["over_refusal_rate"],
        "recall_vanilla_harmful": metrics.get("recall_vanilla_harmful"),
        "recall_adversarial_harmful": metrics.get("recall_adversarial_harmful"),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "extra": {
            "preset": metadata.get("preset", args.preset),
            "search_space_preset": metadata.get("preset", args.preset),
            "embedder": metadata.get("embedder"),
            "embedder_hf_model": hf_emb,
            "embedder_fixed": metadata.get("embedder_fixed", not args.no_fix_embedder),
            "pilot": metadata.get("pilot", args.pilot),
            "model_dir": str(model_dir),
            "data_dir": str(data_dir),
            "results_dir": str(results_dir),
            "run_config": {"mode": args.mode, "n_shots": args.n_shots if args.mode == "fewshot" else None, "seed": args.seed, "preset": args.preset, "pilot": args.pilot, "no_fix_embedder": args.no_fix_embedder},
            "train_metadata": metadata,
            "runtime_environment": _runtime(),
            "timings": {"train_total_sec": (metadata.get("timings") or {}).get("train_total_sec"), "eval_total_sec": float(time.perf_counter() - started)},
            "data_summary": {"train": (metadata.get("data_summary") or {}).get("train"), "test": _test_summary(test_raw, test_file), "eval_binary": _eval_summary(eval_binary, eval_file), "alignment": alignment},
            "prediction_summary": {"n_predictions": int(len(y_pred)), "raw_prediction_values": dict(sorted(raw_pred_values.items())), "raw_none_count": raw_none, "none_policy": "None predictions are treated as jailbreak(label=1)", "pred_safe": int(np.sum(y_pred == 0)), "pred_jailbreak": int(np.sum(y_pred == 1)), "by_data_type": _prediction_by_type(y_true, y_pred, data_types)},
            "eval_counts": eval_counts,
            "decision_module_attrs": decision_attrs,
            "scores_eval_summary": scores_eval_summary,
            "artifact_manifest": _artifact_manifest(model_dir),
        },
    }
    base.save_metrics(result, results_dir)
    base.save_run_metrics_file(result, runs_dir if runs_dir is not None else base.get_runs_dir())

    if getattr(args, "print_hypothesis_log", False):
        hyp = {"model_name": result["model_name"], "mode": result["mode"], "seed": result["seed"], "n_shots": result["n_shots"], "search_space_preset": result["extra"]["search_space_preset"], "metrics_core": {"f1": result["f1"], "recall": result["recall"], "over_refusal_rate": result["over_refusal_rate"], "precision": result["precision"], "recall_vanilla_harmful": result.get("recall_vanilla_harmful"), "recall_adversarial_harmful": result.get("recall_adversarial_harmful")}, "embedder_hf_model": hf_emb, "eval_counts": eval_counts, "prediction_summary": result["extra"]["prediction_summary"], "data_summary": result["extra"]["data_summary"], "scores_eval_summary": scores_eval_summary, "decision_module_attrs": decision_attrs, "artifact_manifest": result["extra"]["artifact_manifest"], "hypothesis_notes": "For threshold/asymmetric-cost conclusions run a separate val threshold sweep; this log fully covers preset/embedder comparison at the trained decision point."}
        print("\n" + "=" * 60)
        print("HYPOTHESIS_LOG (max JSON for Kaggle output)")
        print("=" * 60)
        print(json.dumps(hyp, indent=2, ensure_ascii=False))
    if getattr(args, "print_metrics_json", False):
        print("\n" + "=" * 60)
        print("METRICS_JSON (same object as metrics.json row)")
        print("=" * 60)
        print(json.dumps(result, indent=2, ensure_ascii=False))


base.train = train
base.evaluate = evaluate

if __name__ == "__main__":
    base.main()
'''

p = PROJECT_ROOT / "tasks" / "jailbreak_detection" / "scripts" / "run_autointent_logged.py"
p.write_text(logged_runner, encoding="utf-8")
print("Written", p)


## 4. HF + потоки


In [ ]:
HF_TOKEN = ""
def apply_hf_token():
    t = (HF_TOKEN or "").strip()
    if t:
        os.environ["HF_TOKEN"] = t
        return
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        return
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        import getpass
        os.environ["HF_TOKEN"] = getpass.getpass("HF token: ").strip()
apply_hf_token()
for k, v in {"OMP_NUM_THREADS":"1","OPENBLAS_NUM_THREADS":"1","MKL_NUM_THREADS":"1",
    "VECLIB_MAXIMUM_THREADS":"1","NUMEXPR_NUM_THREADS":"1","LOKY_MAX_CPU_COUNT":"1",
    "TOKENIZERS_PARALLELISM":"false"}.items():
    os.environ.setdefault(k, v)


## 5. Данные


In [ ]:
def run_script(rel, extra=None):
    # -u + PYTHONUNBUFFERED: лог дочернего процесса сразу в stdout Kaggle, без накопления в буфере
    cmd = [sys.executable, "-u", str(PROJECT_ROOT / rel)] + (extra or [])
    print("$", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    # Меньше «красного» stderr: HF / tqdm до import transformers в дочернем процессе
    env.setdefault("TRANSFORMERS_VERBOSITY", "error")
    env.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    env.setdefault("TQDM_DISABLE", "1")
    pp = str(PROJECT_ROOT)
    env["PYTHONPATH"] = pp if not env.get("PYTHONPATH") else pp + os.pathsep + env["PYTHONPATH"]
    subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=env)
run_script("tasks/jailbreak_detection/scripts/prepare_data.py")


In [ ]:
run_script("tasks/jailbreak_detection/scripts/prepare_data.py", ["--full_subset"])


## 5b. Проверка подготовленных данных

Эта ячейка специально падает до обучения, если порядок подготовки данных нарушен или не хватает train/test/eval файлов.

In [ ]:
from pathlib import Path
import json

proc = TASK_ROOT / "data" / "processed"
_SEEDS = (42, 123, 456)
_N_SHOTS = (10, 20, 50)
required = ["test.json", "wildjailbreak_eval_binary.jsonl"]
required += [f"train_shot{n}_seed{s}.json" for n in _N_SHOTS for s in _SEEDS]
required += [f"wildjailbreak_full100k_seed{s}.json" for s in _SEEDS]
missing = [name for name in required if not (proc / name).is_file()]
assert not missing, f"Missing processed data files: {missing}. Run prepare_data.py first, then prepare_data.py --full_subset."

with open(proc / "test.json", encoding="utf-8") as f:
    test = json.load(f)
with open(proc / "train_shot20_seed42.json", encoding="utf-8") as f:
    train20 = json.load(f)

eval_rows = sum(1 for _ in open(proc / "wildjailbreak_eval_binary.jsonl", encoding="utf-8"))
assert len(test["utterances"]) == len(test["labels"]) == eval_rows, (
    len(test["utterances"]), len(test["labels"]), eval_rows
)
assert train20["intents"][0]["name"] == "safe"
assert len(train20["intents"][0]["utterances"]) == 20
assert len(train20["oos_utterances"]) == 20

print("DATA CHECK OK")
print("processed_dir:", proc)
print("test rows:", len(test["utterances"]))
print("train_shot20_seed42: safe=20 jailbreak=20")
print("all required files exist:", required)


## 6. Прогоны: few-shot и full — отдельные ячейки

1. **§6a** — few-shot для **оставшихся** пресетов (`zero-shot-encoders`, `bert-finetune`); classic/nn уже прогнаны отдельно.  
2. **§6b** — таблица по `metrics.json` для строк **не full** (после few).  
3. **§6c** — только **full** (все пресеты × seeds).  
4. **§6d** — таблица по `metrics.json` для строк **`mode == full`**.

Конфиг (`PRESETS`, `SEEDS`, …) продублирован в §6a и §6c — при правках синхронизируй оба. `run_script`: `-u`, тихие env для HF/tqdm, **`--no-automl-progress`**.


In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from tasks.jailbreak_detection.scripts import run_autointent as ra

# §6a — few-shot только для оставшихся тяжёлых пресетов (classic-medium и nn-medium уже готовы).
SEEDS = [42, 123, 456]
N_SHOTS_LIST = [10, 20, 50]
PRESETS = (
    "zero-shot-encoders",
    "bert-finetune",  # CLI-алиас → transformers-light (BERT / transformer FT)
)
USE_PILOT = False
NO_FIX_EMBEDDER = True
RUNNER = "tasks/jailbreak_detection/scripts/run_autointent_logged.py"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
FAILED_LOG = WORK / "failed_runs_jailbreak.jsonl"
REMAINING_METRICS_OUT = WORK / "metrics_jailbreak_fewshot_heavy_remaining.json"

pilot_flag = ["--pilot"] if USE_PILOT else []
auto_flag = ["--no-fix-embedder"] if NO_FIX_EMBEDDER else []


def _log_failed(preset: str, n_shots: int, seed: int, error: str) -> None:
    row = {
        "preset": preset,
        "n_shots": n_shots,
        "seed": seed,
        "error": error,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with FAILED_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("FAILED logged →", FAILED_LOG, flush=True)


print("\n" + "=" * 72, flush=True)
print("§6a — FEWSHOT remaining presets:", PRESETS, flush=True)
print("=" * 72 + "\n", flush=True)

for preset in PRESETS:
    print("\n" + "=" * 72, flush=True)
    print("PRESET:", preset, "— FEWSHOT grid", flush=True)
    print("=" * 72, flush=True)
    for n_shots in N_SHOTS_LIST:
        for seed in SEEDS:
            print(f"\n>>> fewshot preset={preset} n_shots={n_shots} seed={seed}", flush=True)
            sys.stdout.flush()
            try:
                run_script(
                    RUNNER,
                    [
                        "--mode", "fewshot",
                        "--preset", preset,
                        "--n_shots", str(n_shots),
                        "--seed", str(seed),
                        *pilot_flag,
                        *auto_flag,
                        "--no-automl-progress",
                        "--print-metrics-json",
                        "--print-hypothesis-log",
                    ],
                )
            except subprocess.CalledProcessError as exc:
                _log_failed(preset, n_shots, seed, str(exc))
                print(f"SKIP (error): preset={preset} n_shots={n_shots} seed={seed}", flush=True)
            sys.stdout.flush()

print("\n§6a done.", flush=True)

# Сводный JSON только по PRESETS из этой ячейки
metrics_path = TASK_ROOT / "results" / "metrics.json"
remaining_rows = []
if metrics_path.is_file():
    all_rows = json.loads(metrics_path.read_text(encoding="utf-8"))
    preset_set = set(PRESETS)
    for row in all_rows:
        ex = row.get("extra") or {}
        p = ex.get("preset") or ex.get("search_space_preset")
        if row.get("mode") == "full" or p not in preset_set:
            continue
        remaining_rows.append(ra.metrics_row_for_export(row))
    remaining_rows.sort(
        key=lambda r: (str(r.get("preset")), r.get("n_shots") or -1, r.get("seed") or -1)
    )
REMAINING_METRICS_OUT.write_text(
    json.dumps(remaining_rows, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nWrote {len(remaining_rows)} rows → {REMAINING_METRICS_OUT}", flush=True)


### 6b. Сводка `metrics.json` после few-shot

Строки с `mode != "full"` (в т.ч. `10shot`, `20shot`, `50shot`). Запускай сразу после **§6a**.


In [ ]:
import json

p = TASK_ROOT / "results" / "metrics.json"
if not p.is_file():
    print("Нет metrics.json — сначала выполни §6a.", flush=True)
else:
    rows = json.loads(p.read_text(encoding="utf-8"))
    few = [r for r in rows if r.get("mode") != "full"]
    few.sort(
        key=lambda r: (
            str((r.get("extra") or {}).get("search_space_preset") or (r.get("extra") or {}).get("preset") or ""),
            r.get("n_shots") if r.get("n_shots") is not None else -1,
            r.get("seed") if r.get("seed") is not None else -1,
        )
    )
    print("\n" + "=" * 72, flush=True)
    print(f"§6b — метрики после FEWSHOT ({len(few)} строк, mode != 'full')", flush=True)
    print("=" * 72 + "\n", flush=True)
    for r in few:
        ex = r.get("extra") or {}
        pr = ex.get("search_space_preset", ex.get("preset"))
        print(
            f"{r.get('model_name')} | preset={pr} | mode={r.get('mode')} n_shots={r.get('n_shots')} seed={r.get('seed')} | "
            f"F1={r.get('f1'):.4f} R={r.get('recall'):.4f} ORR={r.get('over_refusal_rate'):.4f}",
            flush=True,
        )
    if few:
        print("\n--- Последняя строка few-shot (полный JSON) ---\n", flush=True)
        print(json.dumps(few[-1], indent=2, ensure_ascii=False), flush=True)


### 6c. Full (100K × сиды)

Тот же конфиг пресетов/сидов, что в **§6a**. Запускай после **§6b**.


In [ ]:
import sys

# §6c — только full. Конфиг должен совпадать с §6a.
SEEDS = [42, 123, 456]
PRESETS = (
    "classic-medium",
    "nn-medium",
    "zero-shot-encoders",
    "transformers-light",
)
USE_PILOT = False
NO_FIX_EMBEDDER = True
RUNNER = "tasks/jailbreak_detection/scripts/run_autointent_logged.py"

pilot_flag = ["--pilot"] if USE_PILOT else []
auto_flag = ["--no-fix-embedder"] if NO_FIX_EMBEDDER else []

print("\n" + "#" * 72, flush=True)
print("§6c — FULL only (100K train, wildjailbreak_full100k_seed*.json)", flush=True)
print("#" * 72 + "\n", flush=True)

for preset in PRESETS:
    print("\n" + "#" * 72, flush=True)
    print("PRESET:", preset, "— FULL", flush=True)
    print("#" * 72, flush=True)
    for seed in SEEDS:
        print(f"\n>>> full preset={preset} seed={seed}", flush=True)
        sys.stdout.flush()
        run_script(
            RUNNER,
            [
                "--mode", "full",
                "--preset", preset,
                "--seed", str(seed),
                *pilot_flag,
                *auto_flag,
                "--no-automl-progress",
                "--print-metrics-json",
                "--print-hypothesis-log",
            ],
        )
        sys.stdout.flush()

print("\n§6c done.", flush=True)


### 6d. Сводка `metrics.json` после full

Только строки с `mode == "full"`. Запускай сразу после **§6c**.


In [ ]:
import json

p = TASK_ROOT / "results" / "metrics.json"
if not p.is_file():
    print("Нет metrics.json — сначала §6c.", flush=True)
else:
    rows = json.loads(p.read_text(encoding="utf-8"))
    full_rows = [r for r in rows if r.get("mode") == "full"]
    full_rows.sort(
        key=lambda r: (
            str((r.get("extra") or {}).get("search_space_preset") or (r.get("extra") or {}).get("preset") or ""),
            r.get("seed") if r.get("seed") is not None else -1,
        )
    )
    print("\n" + "=" * 72, flush=True)
    print(f"§6d — метрики после FULL ({len(full_rows)} строк)", flush=True)
    print("=" * 72 + "\n", flush=True)
    for r in full_rows:
        ex = r.get("extra") or {}
        pr = ex.get("search_space_preset", ex.get("preset"))
        print(
            f"{r.get('model_name')} | preset={pr} | mode={r.get('mode')} seed={r.get('seed')} | "
            f"F1={r.get('f1'):.4f} R={r.get('recall'):.4f} ORR={r.get('over_refusal_rate'):.4f}",
            flush=True,
        )
    if full_rows:
        print("\n--- Последняя строка full (полный JSON) ---\n", flush=True)
        print(json.dumps(full_rows[-1], indent=2, ensure_ascii=False), flush=True)


## 7. Сводка `metrics.json` (опционально, всё подряд)

После **§6b** и **§6d** основная сводка уже есть; эта ячейка — хвост последних записей без фильтра по режиму.


In [ ]:
import json
p = TASK_ROOT / "results" / "metrics.json"
rows = json.loads(p.read_text(encoding="utf-8"))
tail = rows[-min(40, len(rows)):]
print("\n=== Сводка последних записей (preset, f1, recall, ORR) ===\n")
for r in tail:
    ex = r.get("extra") or {}
    print(
        f"{r.get('model_name')} | preset={ex.get('search_space_preset', ex.get('preset'))} | "
        f"mode={r.get('mode')} seed={r.get('seed')} | F1={r.get('f1'):.4f} R={r.get('recall'):.4f} ORR={r.get('over_refusal_rate'):.4f}"
    )
print("\nПолный JSON последней строки:\n")
print(json.dumps(tail[-1], indent=2, ensure_ascii=False))


## 8. Сохранение артефактов (Kaggle)

**Почему «пропадают»:** всё под `/kaggle/working` привязано к **текущей сессии**. Новый Run, таймаут, закрытие вкладки или смена машины — и рабочая директория очищается, если не включена персистентность и не сохранён output версии.

**Что сделать вручную в UI Kaggle**

1. **Settings → Persistence** — при наличии опции включи сохранение файлов между перезапусками (ограничения см. в справке Kaggle).
2. **Save Version** — при коммите версии включи опцию, которая **сохраняет output** ноутбука; тогда артефакты из прогона останутся прикреплёнными к этой версии и их можно скачать со страницы версии.

**Ячейка ниже** — собирает весь каталог `kaggle_jailbreak_heavy_presets` в один timestamped `.zip` прямо в `/kaggle/working` и показывает **ссылку для скачивания**. Запусти её, пока сессия ещё жива (после **§6d** или когда нужен архив).

**Уже во время §6a/§6c:** после каждого завершённого прогона скрипт дублирует актуальный **`metrics.json`** в **`/kaggle/working/metrics_jailbreak_latest.json`** — в панели Output видно накопление метрик за ночь без ожидания zip.


In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

# Дублируем логику из §1 на случай одиночного запуска этой ячейки
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
try:
    root = PROJECT_ROOT
except NameError:
    root = WORK / "kaggle_jailbreak_heavy_presets"

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SUTC")
zip_base = WORK / f"jailbreak_heavy_presets_{stamp}"

if not root.exists():
    print("Каталог не найден:", root, "(сначала выполни §1 и прогоны)")
else:
    print("Архивирую", root, "→", f"{zip_base}.zip", flush=True)
    shutil.make_archive(str(zip_base), "zip", root_dir=str(root))
    zpath = zip_base.with_suffix(".zip")
    mib = zpath.stat().st_size / (1024 * 1024)
    print(f"Готово: {zpath} ({mib:.1f} MiB)", flush=True)
    try:
        from IPython.display import FileLink, display

        display(FileLink(str(zpath)))
    except Exception as exc:
        print("FileLink недоступен:", exc)
        print("Скачай файл из панели Output →", zpath.name)

# Лёгкая копия метрик в корень working — удобно искать глазами
try:
    m = TASK_ROOT / "results" / "metrics.json"
    if m.exists():
        dst = WORK / "metrics_jailbreak_latest.json"
        shutil.copy2(m, dst)
        print("Копия metrics.json →", dst, flush=True)
except NameError:
    pass
